# Semi-dynamic Markovian PFE with BPR in the main PFE loop

This version uses the corrected grid network and applies BPR travel-time updating inside the main OD/PFE estimation procedure.


In [2]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Tuple
from collections import deque
import math

import numpy as np
import pandas as pd
from scipy.optimize import minimize, Bounds

In [3]:
@dataclass
class Link:
    link_id: int
    from_node: int
    to_node: int
    capacity: float
    free_flow_time: float

## 1. Input data exactly from the grid-network example

In [4]:
def build_grid_network() -> Dict[int, Link]:
    """Grid-network link data from Chen et al. Table 1."""
    return {
        1: Link(1, 1, 2, 280.0, 2.00),
        2: Link(2, 1, 4, 290.0, 1.50),
        3: Link(3, 1, 5, 280.0, 3.00),
        4: Link(4, 2, 3, 280.0, 1.00),
        5: Link(5, 2, 5, 600.0, 1.00),
        6: Link(6, 3, 6, 300.0, 2.00),
        7: Link(7, 4, 5, 500.0, 2.00),
        8: Link(8, 4, 7, 400.0, 1.00),
        9: Link(9, 5, 6, 500.0, 1.50),
        10: Link(10, 5, 8, 700.0, 1.00),
        11: Link(11, 5, 9, 250.0, 2.00),
        12: Link(12, 6, 9, 300.0, 1.00),
        13: Link(13, 7, 8, 350.0, 1.00),
        14: Link(14, 8, 9, 220.0, 1.00),
    }


def build_true_od() -> Dict[Tuple[int, int], float]:
    """True OD trip table from Chen et al. Table 2."""
    return {
        (1, 6): 120.0,
        (1, 8): 150.0,
        (1, 9): 100.0,
        (2, 6): 130.0,
        (2, 8): 200.0,
        (2, 9): 90.0,
        (4, 6): 80.0,
        (4, 8): 180.0,
        (4, 9): 110.0,
    }


def build_observed_links() -> List[int]:
    """Observed links in Chen et al. Table 1: 3, 5, 6, 7, 9, 10, 11, and 13."""
    return [3, 5, 6, 7, 9, 10, 11, 13]


def build_time_settings() -> Dict[str, float]:
    return {"num_periods": 3, "period_minutes": 10.0}


def build_model_parameters() -> Dict[str, float]:
    """Paper values: theta = 1.50, rho = 11.27."""
    return {"theta": 1.50, "rho": 11.27}


def build_chen_set1_link_flows() -> Dict[int, float]:
    """SUE link-flow Set 1 from Chen et al. Table 1."""
    return {
        1: 124, 2: 137, 3: 109, 4: 77, 5: 467, 6: 77, 7: 212,
        8: 295, 9: 303, 10: 400, 11: 85, 12: 50, 13: 295, 14: 165,
    }


def build_chen_set2_observed_counts() -> Dict[int, float]:
    """Observed link-flow Set 2 from Chen et al. Table 1, available only on measured links."""
    return {3: 108, 5: 495, 6: 82, 7: 236, 9: 285, 10: 390, 11: 70, 13: 296}


def build_grid_scenarios() -> Dict[str, Dict[int, Dict[int, float]]]:
    """
    Scenario data with the corrected observed-link set.

    Important correction from the notebook draft:
    Chen et al. observe link 9, not link 8.
    """
    scenario_1 = {
        1: {3: 109, 5: 467, 6: 77, 7: 212, 9: 303, 10: 400, 11: 85, 13: 295},
        2: {3: 136, 5: 526, 6: 95, 7: 254, 9: 354, 10: 457, 11: 105, 13: 328},
        3: {3: 137, 5: 527, 6: 95, 7: 255, 9: 355, 10: 459, 11: 106, 13: 329},
    }
    scenario_2 = {
        1: {3: 109, 5: 467, 6: 77, 7: 212, 9: 303, 10: 400, 11: 85, 13: 295},
        2: {3: 109, 5: 467, 6: 77, 7: 212, 9: 303, 10: 400, 11: 85, 13: 295},
        3: {3: 109, 5: 467, 6: 77, 7: 212, 9: 303, 10: 400, 11: 85, 13: 295},
    }
    scenario_3 = {
        1: {3: 108, 5: 495, 6: 82, 7: 236, 9: 285, 10: 390, 11: 70, 13: 296},
        2: {3: 108, 5: 495, 6: 82, 7: 236, 9: 285, 10: 390, 11: 70, 13: 296},
        3: {3: 108, 5: 495, 6: 82, 7: 236, 9: 285, 10: 390, 11: 70, 13: 296},
    }
    return {"scenario_1": scenario_1, "scenario_2": scenario_2, "scenario_3": scenario_3}


def build_true_link_flows() -> Dict[int, Dict[int, float]]:
    """
    True link flows used for validation.

    Period 1 equals Chen et al. Table 1 Set 1. Periods 2 and 3 are kept for the
    semi-dynamic notebook workflow, with corrected observed-link indexing.
    """
    return {
        1: build_chen_set1_link_flows(),
        2: {1: 150, 2: 162, 3: 136, 4: 95, 5: 526, 6: 95, 7: 254,
            8: 328, 9: 354, 10: 457, 11: 105, 12: 61, 13: 328, 14: 189},
        3: {1: 150, 2: 162, 3: 137, 4: 95, 5: 527, 6: 95, 7: 255,
            8: 329, 9: 355, 10: 459, 11: 106, 12: 61, 13: 329, 14: 189},
    }


## 2. Network utilities

In [5]:
def get_nodes_from_links(links: Dict[int, Link]) -> List[int]:
    nodes = set()
    for link in links.values():
        nodes.add(link.from_node)
        nodes.add(link.to_node)
    return sorted(nodes)


def build_adjacency(links: Dict[int, Link]):
    outgoing: Dict[int, List[int]] = {}
    incoming: Dict[int, List[int]] = {}
    for lid, link in links.items():
        outgoing.setdefault(link.from_node, []).append(lid)
        incoming.setdefault(link.to_node, []).append(lid)
        outgoing.setdefault(link.to_node, [])
        incoming.setdefault(link.from_node, [])
    return outgoing, incoming


def topological_order(links: Dict[int, Link]) -> List[int]:
    nodes = get_nodes_from_links(links)
    outgoing, incoming = build_adjacency(links)
    indegree = {node: len(incoming.get(node, [])) for node in nodes}
    q = deque([node for node in nodes if indegree[node] == 0])
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for lid in outgoing.get(u, []):
            v = links[lid].to_node
            indegree[v] -= 1
            if indegree[v] == 0:
                q.append(v)
    if len(order) != len(nodes):
        raise ValueError("Network is not acyclic; topological order failed.")
    return order


def safe_xlogx(x: float) -> float:
    if x <= 0:
        return 0.0
    return x * math.log(x)

## 3. Absorbing-Markov assignment using matrix value formulation

This section follows the paper's matrix idea:

\[
V = W + W^2 + W^3 + \cdots = (I-W)^{-1} - I
\]

where each nonzero element of \(W\) is \(\exp(-	heta c_{ij})\). For one destination \(s\), the downstream value from node \(i\) is the element \(V_{is}\), and transition probabilities are computed using the logit/Markov formula.

In [6]:
def build_W_matrix(
    links: Dict[int, Link],
    theta: float,
    generalized_costs: Dict[int, float] | None = None,
):
    if generalized_costs is None:
        generalized_costs = {lid: links[lid].free_flow_time for lid in links}
    nodes = get_nodes_from_links(links)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    n = len(nodes)
    W = np.zeros((n, n), dtype=float)
    for lid, link in links.items():
        i = node_to_idx[link.from_node]
        j = node_to_idx[link.to_node]
        W[i, j] += math.exp(-theta * generalized_costs[lid])
    return W, nodes, node_to_idx


def compute_markov_value_matrix(
    links: Dict[int, Link],
    theta: float,
    generalized_costs: Dict[int, float] | None = None,
):
    W, nodes, node_to_idx = build_W_matrix(links, theta, generalized_costs)
    I = np.eye(len(nodes))
    # Paper Eq. (36): V = (I-W)^-1 - I for the acyclic grid network.
    V = np.linalg.inv(I - W) - I
    return V, W, nodes, node_to_idx


def compute_transition_probabilities_matrix(
    links: Dict[int, Link],
    theta: float,
    destination: int,
    generalized_costs: Dict[int, float] | None = None,
) -> Tuple[Dict[int, float], Dict[int, float]]:
    if generalized_costs is None:
        generalized_costs = {lid: links[lid].free_flow_time for lid in links}

    V, W, nodes, node_to_idx = compute_markov_value_matrix(links, theta, generalized_costs)
    s_idx = node_to_idx[destination]

    A = {}
    for node in nodes:
        idx = node_to_idx[node]
        A[node] = 1.0 if node == destination else max(V[idx, s_idx], 0.0)

    probs: Dict[int, float] = {}
    outgoing, _ = build_adjacency(links)
    for i_node in nodes:
        if i_node == destination:
            continue
        denom = A[i_node]
        if denom <= 0:
            continue
        for lid in outgoing.get(i_node, []):
            j_node = links[lid].to_node
            downstream = 1.0 if j_node == destination else A[j_node]
            if downstream <= 0:
                continue
            numer = math.exp(-theta * generalized_costs[lid]) * downstream
            probs[lid] = numer / denom

    return probs, A


def assign_single_od_absorbing_markov(
    links: Dict[int, Link],
    origin: int,
    destination: int,
    demand: float,
    theta: float,
    generalized_costs: Dict[int, float] | None = None,
):
    if generalized_costs is None:
        generalized_costs = {lid: links[lid].free_flow_time for lid in links}

    probs, A = compute_transition_probabilities_matrix(
        links=links,
        theta=theta,
        destination=destination,
        generalized_costs=generalized_costs,
    )

    order = topological_order(links)
    outgoing, _ = build_adjacency(links)
    node_flow = {node: 0.0 for node in get_nodes_from_links(links)}
    link_flow = {lid: 0.0 for lid in links}
    node_flow[origin] = demand

    for i in order:
        if i == destination:
            continue
        inflow = node_flow[i]
        if inflow <= 0:
            continue
        for lid in outgoing.get(i, []):
            if lid not in probs:
                continue
            f = inflow * probs[lid]
            j = links[lid].to_node
            link_flow[lid] += f
            node_flow[j] += f

    return link_flow, node_flow, probs, A


def assign_all_od_absorbing_markov_by_origin(
    links: Dict[int, Link],
    od: Dict[Tuple[int, int], float],
    theta: float,
    generalized_costs: Dict[int, float] | None = None,
):
    if generalized_costs is None:
        generalized_costs = {lid: links[lid].free_flow_time for lid in links}

    origins = sorted({r for r, _ in od})
    total_link_flows = {lid: 0.0 for lid in links}
    origin_link_flows = {r: {lid: 0.0 for lid in links} for r in origins}

    for (r, s), q_rs in od.items():
        lf, _, _, _ = assign_single_od_absorbing_markov(
            links=links,
            origin=r,
            destination=s,
            demand=q_rs,
            theta=theta,
            generalized_costs=generalized_costs,
        )
        for lid, flow in lf.items():
            total_link_flows[lid] += flow
            origin_link_flows[r][lid] += flow

    return total_link_flows, origin_link_flows

## 4. Evaluation functions

In [7]:
def compute_rmsep(estimated: Dict[int, float], true_values: Dict[int, float], link_ids: List[int] | None = None) -> float:
    if link_ids is None:
        link_ids = sorted(true_values.keys())
    squared_pct_errors = []
    for lid in link_ids:
        x_hat = estimated[lid]
        x_true = true_values[lid]
        if x_true == 0:
            continue
        squared_pct_errors.append(((x_hat - x_true) / x_true) ** 2)
    return 100.0 * math.sqrt(sum(squared_pct_errors) / len(squared_pct_errors))


def compute_od_rmsep(estimated_od: Dict[Tuple[int, int], float], true_od: Dict[Tuple[int, int], float]) -> float:
    squared_pct_errors = []
    for od_pair, true_value in true_od.items():
        if true_value == 0:
            continue
        est_value = estimated_od[od_pair]
        squared_pct_errors.append(((est_value - true_value) / true_value) ** 2)
    return 100.0 * math.sqrt(sum(squared_pct_errors) / len(squared_pct_errors))


def compute_r2_od(estimated_od, true_od):
    y_true = np.array([true_od[k] for k in true_od])
    y_pred = np.array([estimated_od[k] for k in true_od])
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - ss_res / ss_tot

## 5. validation: use given OD and run assignment with the paper BPR link-performance function

In [8]:
def bpr_travel_time(link: Link, flow: float, alpha: float = 0.15, beta: float = 4.0) -> float:
    """Chen et al. Eq. (32): t_a = t_a^f * (1 + 0.15 * (x_a / C_a)^4)."""
    return link.free_flow_time * (1.0 + alpha * (flow / link.capacity) ** beta)


def update_bpr_travel_times(
    links: Dict[int, Link],
    flows: Dict[int, float],
    alpha: float = 0.15,
    beta: float = 4.0,
) -> Dict[int, float]:
    """Return BPR travel times for all links using the current assigned flows."""
    return {
        lid: bpr_travel_time(links[lid], flows.get(lid, 0.0), alpha=alpha, beta=beta)
        for lid in links
    }


def assign_true_od_free_flow_validation():
    """Diagnostic only: assign true OD using free-flow travel times."""
    links = build_grid_network()
    true_od = build_true_od()
    true_link_flows = build_true_link_flows()[1]
    theta = build_model_parameters()["theta"]

    flows, _ = assign_all_od_absorbing_markov_by_origin(
        links=links,
        od=true_od,
        theta=theta,
    )

    rows = []
    for lid in sorted(links):
        paper_true = true_link_flows[lid]
        est = flows[lid]
        rows.append({
            "Link": lid,
            "Estimated flow": round(est, 2),
            "Paper Set 1 flow": paper_true,
            "Difference": round(est - paper_true, 2),
            "Percent error (%)": round(100.0 * (est - paper_true) / paper_true, 2),
        })

    df = pd.DataFrame(rows)
    print("===== TRUE OD -> LINK FLOWS USING FREE-FLOW COSTS =====")
    print(df.to_string(index=False))
    print(f"\nRMSEP (all links): {compute_rmsep(flows, true_link_flows):.2f}")
    return flows, df


def assign_true_od_bpr_sue_validation(
    max_iter: int = 10000,
    tol: float = 1e-9,
):
    """
    Paper-consistent validation: use true OD and solve logit-based SUE with BPR link costs.

    This is the validation that matches Chen et al. Table 1 Set 1 most closely.
    The paper states Set 1 was synthesized according to a logit-based SUE model
    with theta = 1.50 and uses the BPR travel-time function.
    """
    links = build_grid_network()
    true_od = build_true_od()
    true_link_flows = build_true_link_flows()[1]
    theta = build_model_parameters()["theta"]

    flows = {lid: 0.0 for lid in links}

    for it in range(1, max_iter + 1):
        costs = update_bpr_travel_times(links, flows)
        auxiliary_flows, _ = assign_all_od_absorbing_markov_by_origin(
            links=links,
            od=true_od,
            theta=theta,
            generalized_costs=costs,
        )

        step_size = 1.0 / it  # Method of successive averages (MSA)
        new_flows = {
            lid: flows[lid] + step_size * (auxiliary_flows[lid] - flows[lid])
            for lid in links
        }

        max_change = max(abs(new_flows[lid] - flows[lid]) for lid in links)
        flows = new_flows
        if max_change < tol:
            break

    rows = []
    for lid in sorted(links):
        paper_true = true_link_flows[lid]
        est = flows[lid]
        rows.append({
            "Link": lid,
            "Estimated flow": round(est, 2),
            "Paper Set 1 flow": paper_true,
            "Difference": round(est - paper_true, 2),
            "Percent error (%)": round(100.0 * (est - paper_true) / paper_true, 2),
        })

    df = pd.DataFrame(rows)
    print("===== TRUE OD -> LINK FLOWS USING BPR LOGIT-SUE =====")
    print(f"Converged/stopped after {it} MSA iterations")
    print(df.to_string(index=False))
    print(f"\nRMSEP (all links): {compute_rmsep(flows, true_link_flows):.2f}")
    return flows, df


# Run both validations so you can show the difference.
free_flow_validation_flows, free_flow_validation_table = assign_true_od_free_flow_validation()
print("\n")
bpr_sue_validation_flows, bpr_sue_validation_table = assign_true_od_bpr_sue_validation()


===== TRUE OD -> LINK FLOWS USING FREE-FLOW COSTS =====
 Link  Estimated flow  Paper Set 1 flow  Difference  Percent error (%)
    1          124.57               124        0.57               0.46
    2          142.74               137        5.74               4.19
    3          102.68               109       -6.32              -5.79
    4           71.05                77       -5.95              -7.73
    5          473.53               467        6.53               1.40
    6           71.05                77       -5.95              -7.73
    7          200.45               212      -11.55              -5.45
    8          312.29               295       17.29               5.86
    9          306.10               303        3.10               1.02
   10          392.10               400       -7.90              -1.97
   11           78.46                85       -6.54              -7.70
   12           47.15                50       -2.85              -5.70
   13          312.29

### Important correction made

The previous validation used free-flow travel times only. Chen et al. state that the grid-network Set 1 flows were synthesized using a logit-based SUE model and the BPR link travel-time function. Therefore, to validate the paper's Table 1 Set 1 flows, the correct test is **true OD + BPR logit-SUE assignment**, not only true OD + free-flow assignment.

## 6. Semi-dynamic residual and generalized costs

In [9]:
def shortest_path_cost_dag(
    links: Dict[int, Link],
    origin: int,
    destination: int,
    generalized_costs: Dict[int, float] | None = None,
) -> float:
    if generalized_costs is None:
        generalized_costs = {lid: links[lid].free_flow_time for lid in links}

    outgoing, _ = build_adjacency(links)
    order = topological_order(links)
    dist = {node: float("inf") for node in get_nodes_from_links(links)}
    dist[origin] = 0.0

    for i in order:
        if dist[i] == float("inf"):
            continue
        for lid in outgoing.get(i, []):
            j = links[lid].to_node
            new_dist = dist[i] + generalized_costs[lid]
            if new_dist < dist[j]:
                dist[j] = new_dist
    return dist[destination]


def compute_residual_from_q(q: Dict[Tuple[int, int], float], links: Dict[int, Link], Tw: float, generalized_costs=None):
    residual = {}
    for (r, s), q_rs in q.items():
        C_rs = shortest_path_cost_dag(links, r, s, generalized_costs)
        residual[(r, s)] = (C_rs / (2.0 * Tw)) * q_rs
    return residual


def make_generalized_costs(
    links: Dict[int, Link],
    observed_links: List[int],
    l: Dict[int, float],
    u: Dict[int, float],
    d: Dict[int, float],
    t_tilde: Dict[int, float] | None = None,
) -> Dict[int, float]:
    if t_tilde is None:
        t_tilde = {lid: links[lid].free_flow_time for lid in links}
    observed_set = set(observed_links)
    gen_costs = {}
    for lid in links:
        if lid in observed_set:
            gen_costs[lid] = t_tilde[lid] - l.get(lid, 0.0) - u.get(lid, 0.0)
        else:
            gen_costs[lid] = t_tilde[lid] - d.get(lid, 0.0)
        # Numerical protection only; without it exp(-theta*c) may overflow if c becomes nonpositive.
        gen_costs[lid] = max(gen_costs[lid], 1e-9)
    return gen_costs

## 7. Iterative balancing subproblem

In [10]:
def solve_subproblem_iterative_balancing(
    links: Dict[int, Link],
    q_effective: Dict[Tuple[int, int], float],
    theta: float,
    observed_links: List[int],
    observed_counts: Dict[int, float],
    t_tilde: Dict[int, float] | None = None,
    eta0: float = 1e-6,
    eta: float = 1e-5,
    max_inner_iter: int = 200,
) -> Dict:
    all_links = sorted(links.keys())
    observed_set = set(observed_links)
    unobserved_links = [lid for lid in all_links if lid not in observed_set]

    if t_tilde is None:
        t_tilde = {lid: links[lid].free_flow_time for lid in links}

    l = {lid: (eta0 if lid in observed_set else 0.0) for lid in all_links}
    u = {lid: 0.0 for lid in all_links}
    d = {lid: 0.0 for lid in all_links}

    flows = {lid: 0.0 for lid in all_links}
    origin_link_flows = {}
    psi = {lid: 0.0 for lid in observed_links}
    last_xi = float("inf")

    for m in range(1, max_inner_iter + 1):
        l_old = l.copy()
        u_old = u.copy()
        d_old = d.copy()

        gen_costs = make_generalized_costs(links, observed_links, l, u, d, t_tilde)

        flows, origin_link_flows = assign_all_od_absorbing_markov_by_origin(
            links=links,
            od=q_effective,
            theta=theta,
            generalized_costs=gen_costs,
        )

        psi = {lid: abs(observed_counts[lid] - flows[lid]) for lid in observed_links}

        for lid in unobserved_links:
            x_lid = max(flows[lid], 1e-12)
            cap = max(links[lid].capacity, 1e-12)
            lam = (1.0 / theta) * math.log(cap / x_lid)
            d[lid] = min(0.0, d[lid] + lam)

        for lid in observed_links:
            x_lid = max(flows[lid], 1e-12)
            v_lid = max(observed_counts[lid], 1e-12)
            psi_lid = max(psi[lid], 1e-12)
            beta = (1.0 / theta) * math.log(v_lid / (x_lid + psi_lid))
            pi = (1.0 / theta) * math.log(
                (v_lid + math.sqrt(v_lid**2 + 4.0 * x_lid * psi_lid)) / (2.0 * x_lid)
            )
            l[lid] = max(0.0, l[lid] + beta)
            u[lid] = min(0.0, u[lid] + pi)

        xi = max(
            max(abs(l[lid] - l_old[lid]) for lid in all_links),
            max(abs(u[lid] - u_old[lid]) for lid in all_links),
            max(abs(d[lid] - d_old[lid]) for lid in all_links),
        )
        last_xi = xi
        if xi < eta:
            break

    return {
        "flows": flows,
        "origin_link_flows": origin_link_flows,
        "psi": psi,
        "l": l,
        "u": u,
        "d": d,
        "generalized_costs": make_generalized_costs(links, observed_links, l, u, d, t_tilde),
        "inner_iterations": m,
        "inner_xi": last_xi,
    }

## 8. PFE objective for fixed q and outer OD estimation

In [11]:
def solve_pfe_subproblem_with_bpr_outer_loop(
    links: Dict[int, Link],
    q_effective: Dict[Tuple[int, int], float],
    theta: float,
    observed_links: List[int],
    observed_counts: Dict[int, float],
    max_outer_iter: int = 50,
    outer_tol: float = 1e-6,
    max_inner_iter: int = 200,
) -> Dict:
    """
    Main PFE network-loading solver with BPR travel-time updating.

    This adds the paper-style partial-linearization outer loop:
        1. Fix link travel times t_tilde.
        2. Solve Sub-PFE/Norm-link by iterative balancing.
        3. Update travel times using the BPR link performance function t_a(x_a).
        4. Repeat until link flows stabilize.

    The absorbing Markov assignment remains inside solve_subproblem_iterative_balancing().
    """
    all_links = sorted(links.keys())

    # Start from zero flow, so the first BPR travel time equals free-flow travel time.
    prev_flows = {lid: 0.0 for lid in all_links}
    t_tilde = update_bpr_travel_times(links, prev_flows)

    sub = None
    outer_gap = float("inf")

    for n in range(1, max_outer_iter + 1):
        # Inner problem: iterative balancing + absorbing Markov assignment
        sub = solve_subproblem_iterative_balancing(
            links=links,
            q_effective=q_effective,
            theta=theta,
            observed_links=observed_links,
            observed_counts=observed_counts,
            t_tilde=t_tilde,
            max_inner_iter=max_inner_iter,
        )

        flows = sub["flows"]
        outer_gap = max(abs(flows[lid] - prev_flows[lid]) for lid in all_links)

        # Update BPR travel times using the newly estimated link flows.
        new_t_tilde = update_bpr_travel_times(links, flows)

        if outer_gap < outer_tol:
            t_tilde = new_t_tilde
            break

        prev_flows = flows.copy()
        t_tilde = new_t_tilde

    sub["bpr_travel_times"] = t_tilde
    sub["outer_iterations"] = n
    sub["outer_gap"] = outer_gap

    return sub


def link_entropy_term(links: Dict[int, Link], origin_link_flows: Dict[int, Dict[int, float]]) -> float:
    _, incoming = build_adjacency(links)
    nodes = get_nodes_from_links(links)
    total = 0.0
    for r, xr in origin_link_flows.items():
        for node in nodes:
            incoming_flow = sum(xr[lid] for lid in incoming.get(node, []))
            total += safe_xlogx(incoming_flow)
        for lid in links:
            total -= safe_xlogx(xr[lid])
    return total


def subproblem_objective_value(
    links: Dict[int, Link],
    theta: float,
    rho: float,
    sub: Dict,
    t_tilde: Dict[int, float] | None = None,
) -> float:
    if t_tilde is None:
        t_tilde = {lid: links[lid].free_flow_time for lid in links}

    flows = sub["flows"]
    origin_link_flows = sub["origin_link_flows"]
    psi = sub["psi"]

    linearized_travel = sum(t_tilde[lid] * flows[lid] for lid in links)
    entropy = link_entropy_term(links, origin_link_flows)
    virtual_entropy = sum(psi_a * math.log(max(psi_a, 1e-12)) - psi_a for psi_a in psi.values())
    virtual_penalty = rho * sum(psi.values())
    return linearized_travel + (1.0 / theta) * entropy + (1.0 / theta) * virtual_entropy + virtual_penalty


def pfe_objective_for_q(
    q_vector: np.ndarray,
    od_pairs: List[Tuple[int, int]],
    links: Dict[int, Link],
    theta: float,
    rho: float,
    observed_links: List[int],
    observed_counts: Dict[int, float],
    residual_prev: Dict[Tuple[int, int], float],
) -> float:
    q = {od_pairs[i]: max(float(q_vector[i]), 1e-9) for i in range(len(od_pairs))}
    q_effective = {od_pair: q[od_pair] + residual_prev.get(od_pair, 0.0) for od_pair in od_pairs}

    # Main change: solve Sub-PFE with BPR travel-time updating.
    sub = solve_pfe_subproblem_with_bpr_outer_loop(
        links=links,
        q_effective=q_effective,
        theta=theta,
        observed_links=observed_links,
        observed_counts=observed_counts,
    )

    return subproblem_objective_value(
        links=links,
        theta=theta,
        rho=rho,
        sub=sub,
        t_tilde=sub["bpr_travel_times"],
    )


def estimate_period_q(
    links: Dict[int, Link],
    od_initial: Dict[Tuple[int, int], float],
    theta: float,
    rho: float,
    observed_links: List[int],
    observed_counts: Dict[int, float],
    residual_prev: Dict[Tuple[int, int], float],
    Tw: float,
    max_factor: float = 3.0,
) -> Dict:
    od_pairs = list(od_initial.keys())
    x0 = np.array([od_initial[od_pair] for od_pair in od_pairs], dtype=float)
    lower = np.zeros(len(od_pairs))
    upper = np.array([od_initial[od_pair] * max_factor for od_pair in od_pairs], dtype=float)

    def objective(x):
        return pfe_objective_for_q(
            q_vector=x,
            od_pairs=od_pairs,
            links=links,
            theta=theta,
            rho=rho,
            observed_links=observed_links,
            observed_counts=observed_counts,
            residual_prev=residual_prev,
        )

    opt = minimize(
        objective,
        x0,
        method="L-BFGS-B",
        bounds=list(zip(lower, upper)),
        options={"maxiter": 300, "ftol": 1e-8},
    )

    q_est = {od_pairs[i]: float(opt.x[i]) for i in range(len(od_pairs))}
    q_effective = {od_pair: q_est[od_pair] + residual_prev.get(od_pair, 0.0) for od_pair in od_pairs}

    # Final BPR-PFE solve for the optimized q.
    sub = solve_pfe_subproblem_with_bpr_outer_loop(
        links=links,
        q_effective=q_effective,
        theta=theta,
        observed_links=observed_links,
        observed_counts=observed_counts,
    )

    # Residual flow should be based on the post-assignment BPR travel times, not free-flow times.
    residual_next = compute_residual_from_q(
        q=q_est,
        links=links,
        Tw=Tw,
        generalized_costs=sub["bpr_travel_times"],
    )

    return {
        "q": q_est,
        "q_effective": q_effective,
        "flows": sub["flows"],
        "origin_link_flows": sub["origin_link_flows"],
        "psi": sub["psi"],
        "l": sub["l"],
        "u": sub["u"],
        "d": sub["d"],
        "generalized_costs": sub["generalized_costs"],
        "bpr_travel_times": sub["bpr_travel_times"],
        "inner_iterations": sub["inner_iterations"],
        "inner_xi": sub["inner_xi"],
        "outer_iterations": sub["outer_iterations"],
        "outer_gap": sub["outer_gap"],
        "residual_next": residual_next,
        "objective": float(opt.fun),
        "success": bool(opt.success),
        "message": opt.message,
    }



## 9. Semi-dynamic scenario runner

In [12]:
def run_scenario_paper_exact(
    scenario_name: str,
    links: Dict[int, Link],
    od_base: Dict[Tuple[int, int], float],
    theta: float,
    rho: float,
    observed_links: List[int],
    scenarios: Dict[str, Dict[int, Dict[int, float]]],
    Tw: float,
) -> Dict[int, Dict]:
    results = {}
    residual_prev = {od_pair: 0.0 for od_pair in od_base}
    q_initial = od_base.copy()

    for tau in [1, 2, 3]:
        observed_counts = scenarios[scenario_name][tau]
        period_result = estimate_period_q(
            links=links,
            od_initial=q_initial,
            theta=theta,
            rho=rho,
            observed_links=observed_links,
            observed_counts=observed_counts,
            residual_prev=residual_prev,
            Tw=Tw,
        )
        results[tau] = period_result
        residual_prev = period_result["residual_next"]
        q_initial = period_result["q"]
    return results


def print_scenario_summary(scenario_name: str, results: Dict[int, Dict], true_od: Dict[Tuple[int,int], float]):
    print(f"\n==============================")
    print(f"{scenario_name.upper()} SUMMARY")
    print(f"==============================")
    for tau in sorted(results):
        r = results[tau]
        print(f"\nTime Period {tau}")
        print(f"  Optimizer success: {r['success']}")
        print(f"  Objective: {r['objective']:.2f}")
        print(f"  Inner iterations: {r['inner_iterations']}")
        print(f"  Inner xi: {r['inner_xi']:.6g}")
        print(f"  Total virtual flow psi: {sum(r['psi'].values()):.2f}")
        print(f"  Estimated q total: {sum(r['q'].values()):.2f}")
        print(f"  Effective q total q+residual_prev: {sum(r['q_effective'].values()):.2f}")
        print(f"  Residual to next period: {sum(r['residual_next'].values()):.2f}")
        print(f"  Link-flow sum: {sum(r['flows'].values()):.2f}")
        print(f"  OD RMSEP: {compute_od_rmsep(r['q'], true_od):.2f}")


def print_link_rmsep_summary(results, true_flows, observed_links):
    print("\n===== PERIOD-BY-PERIOD LINK RMSEP =====")
    for tau in sorted(results.keys()):
        flows = results[tau]["flows"]
        rmsep_all = compute_rmsep(flows, true_flows[tau])
        rmsep_obs = compute_rmsep(flows, true_flows[tau], observed_links)
        unobserved = [lid for lid in true_flows[tau] if lid not in observed_links]
        rmsep_unobs = compute_rmsep(flows, true_flows[tau], unobserved)
        print(f"\nTime Period {tau}")
        print(f"  RMSEP (all): {rmsep_all:.2f}")
        print(f"  RMSEP (observed): {rmsep_obs:.2f}")
        print(f"  RMSEP (unobserved): {rmsep_unobs:.2f}")


## 10. Run the paper-described full model

This may take time because each OD optimization repeatedly solves the iterative-balancing subproblem.

In [13]:
def main():
    links = build_grid_network()
    od_base = build_true_od()
    observed_links = build_observed_links()
    settings = build_time_settings()
    params = build_model_parameters()
    scenarios = build_grid_scenarios()
    true_flows = build_true_link_flows()

    theta = params["theta"]
    rho = params["rho"]
    Tw = settings["period_minutes"]

    print("===== INITIAL CHECK =====")
    print("Number of links:", len(links))
    print("Number of OD pairs:", len(od_base))
    print("Observed links:", observed_links)
    print("Time settings:", settings)
    print("Model parameters:", params)

    all_results = {}
    for scenario_name in ["scenario_1", "scenario_2", "scenario_3"]:
        print("\n\n==============================")
        print(f"RUNNING {scenario_name.upper()}")
        print("==============================")
        results = run_scenario_paper_exact(
            scenario_name=scenario_name,
            links=links,
            od_base=od_base,
            theta=theta,
            rho=rho,
            observed_links=observed_links,
            scenarios=scenarios,
            Tw=Tw,
        )
        all_results[scenario_name] = results
        print_scenario_summary(scenario_name, results, od_base)
        print_link_rmsep_summary(results, true_flows, observed_links)

    return all_results

# Uncomment to run full model:
all_results = main()


===== INITIAL CHECK =====
Number of links: 14
Number of OD pairs: 9
Observed links: [3, 5, 6, 7, 9, 10, 11, 13]
Time settings: {'num_periods': 3, 'period_minutes': 10.0}
Model parameters: {'theta': 1.5, 'rho': 11.27}


RUNNING SCENARIO_1

SCENARIO_1 SUMMARY

Time Period 1
  Optimizer success: True
  Objective: 4479.16
  Inner iterations: 1
  Inner xi: 5.81843e-08
  Total virtual flow psi: 0.08
  Estimated q total: 1159.93
  Effective q total q+residual_prev: 1159.93
  Residual to next period: 179.87
  Link-flow sum: 2796.28
  OD RMSEP: 0.54

Time Period 2
  Optimizer success: True
  Objective: 5332.70
  Inner iterations: 1
  Inner xi: 3.99915e-08
  Total virtual flow psi: 0.78
  Estimated q total: 1158.40
  Effective q total q+residual_prev: 1338.27
  Residual to next period: 183.23
  Link-flow sum: 3237.52
  OD RMSEP: 0.62

Time Period 3
  Optimizer success: True
  Objective: 5361.11
  Inner iterations: 1
  Inner xi: 2.5369e-07
  Total virtual flow psi: 1.36
  Estimated q total: 1159.

## 11. Paper-style tables after running `all_results = main()`

In [14]:
def build_paper_table4_like(all_results, true_od):
    rows = []
    for scenario_name, results in all_results.items():
        rmsep_row = {"Scenario": scenario_name, "Metric": "RMSEP"}
        r2_row = {"Scenario": scenario_name, "Metric": "Coefficient of determination"}
        for tau in [1, 2, 3]:
            q = results[tau]["q"]
            rmsep_row[f"TP{tau}"] = round(compute_od_rmsep(q, true_od), 2)
            r2_row[f"TP{tau}"] = round(compute_r2_od(q, true_od), 3)
        rows.append(rmsep_row)
        rows.append(r2_row)
    return pd.DataFrame(rows)


def build_paper_table5_like(all_results, true_od):
    true_total = sum(true_od.values())
    rows = []
    for scenario_name, results in all_results.items():
        est_row = {"Scenario": scenario_name, "Metric": "Total OD Volume (estimated)"}
        true_row = {"Scenario": scenario_name, "Metric": "Total OD Volume (true)"}
        for tau in [1, 2, 3]:
            est_row[f"TP{tau}"] = round(sum(results[tau]["q"].values()), 1)
            true_row[f"TP{tau}"] = round(true_total, 1)
        rows.append(est_row)
        rows.append(true_row)
    return pd.DataFrame(rows)


def build_link_rmsep_table(all_results, true_flows, observed_links):
    links = build_grid_network()
    rows = []
    for scenario_name, results in all_results.items():
        for tau in [1, 2, 3]:
            flows = results[tau]["flows"]
            unobserved = [lid for lid in links if lid not in observed_links]
            rows.append({
                "Scenario": scenario_name,
                "TP": tau,
                "RMSEP all": round(compute_rmsep(flows, true_flows[tau]), 2),
                "RMSEP observed": round(compute_rmsep(flows, true_flows[tau], observed_links), 2),
                "RMSEP unobserved": round(compute_rmsep(flows, true_flows[tau], unobserved), 2),
            })
    return pd.DataFrame(rows)

# ============================================================
# RUN TABLES
# ============================================================

true_od = build_true_od()
true_flows = build_true_link_flows()
observed_links = build_observed_links()

table4 = build_paper_table4_like(all_results, true_od)
table5 = build_paper_table5_like(all_results, true_od)

link_table = build_link_rmsep_table(
    all_results,
    true_flows,
    observed_links
)

print("\n===== PAPER TABLE 4 =====")
display(table4)

print("\n===== PAPER TABLE 5 =====")
display(table5)

print("\n===== LINK RMSEP TABLE =====")
display(link_table)


===== PAPER TABLE 4 =====


,Scenario,Metric,TP1,TP2,TP3
0,scenario_1,RMSEP,0.540,0.620,0.800
1,scenario_1,Coefficient of determination,1.000,1.000,0.999
2,scenario_2,RMSEP,0.540,16.890,13.770
3,scenario_2,Coefficient of determination,1.000,0.712,0.802
4,scenario_3,RMSEP,60.480,55.330,55.820
5,scenario_3,Coefficient of determination,-2.397,-1.943,-1.970



===== PAPER TABLE 5 =====


,Scenario,Metric,TP1,TP2,TP3
0,scenario_1,Total OD Volume (estimated),1159.9,1158.4,1159.4
1,scenario_1,Total OD Volume (true),1160.0,1160.0,1160.0
2,scenario_2,Total OD Volume (estimated),1159.9,979.8,1010.2
3,scenario_2,Total OD Volume (true),1160.0,1160.0,1160.0
4,scenario_3,Total OD Volume (estimated),1120.8,970.5,991.7
5,scenario_3,Total OD Volume (true),1160.0,1160.0,1160.0



===== LINK RMSEP TABLE =====


,Scenario,TP,RMSEP all,RMSEP observed,RMSEP unobserved
0,scenario_1,1,0.12,0.03,0.18
1,scenario_1,2,0.35,0.25,0.45
2,scenario_1,3,0.42,0.36,0.49
3,scenario_2,1,0.12,0.03,0.18
4,scenario_2,2,15.79,15.78,15.81
5,scenario_2,3,16.02,16.13,15.86
6,scenario_3,1,39.44,31.48,48.05
7,scenario_3,2,43.79,35.76,52.62
8,scenario_3,3,43.91,35.95,52.69


In [15]:
# ============================================================
# SENSITIVITY OF BPR / LOGIT-SUE ASSIGNMENT TO STARTING POINT
# TRUE OD ONLY — NO OD ESTIMATION
# ============================================================

import pandas as pd
import numpy as np


def bpr_time(link, flow, alpha=0.15, beta=4.0):
    return link.free_flow_time * (1.0 + alpha * (flow / link.capacity) ** beta)


def compute_bpr_costs_from_flows(links, flows):
    return {
        lid: bpr_time(links[lid], flows.get(lid, 0.0))
        for lid in links
    }


def run_bpr_logit_equilibrium_assignment(
    links,
    od,
    theta,
    initial_flows=None,
    max_iter=5000,
    tol=1e-8,
    order_name="default",
):
    """
    True OD assignment only:
    OD -> Markov/logit assignment -> BPR cost update -> repeat with MSA.
    """

    # Initial flow
    if initial_flows is None:
        flows = {lid: 0.0 for lid in links}
    else:
        flows = initial_flows.copy()

    history = []

    for it in range(1, max_iter + 1):

        # Step 1: compute BPR costs from current flows
        costs = compute_bpr_costs_from_flows(links, flows)

        # Step 2: assign true OD using current BPR costs
        aux_flows, _ = assign_all_od_absorbing_markov_by_origin(
            links=links,
            od=od,
            theta=theta,
            generalized_costs=costs,
        )

        # Step 3: MSA update
        step = 1.0 / it
        new_flows = {
            lid: flows[lid] + step * (aux_flows[lid] - flows[lid])
            for lid in links
        }

        max_diff = max(abs(new_flows[lid] - flows[lid]) for lid in links)
        history.append(max_diff)

        flows = new_flows

        if max_diff < tol:
            break

    return {
        "name": order_name,
        "flows": flows,
        "iterations": it,
        "last_diff": max_diff,
        "history": history,
    }


def build_initial_flow_cases(links, od, theta):
    """
    Different starting points for equilibrium assignment.
    """

    zero_flows = {lid: 0.0 for lid in links}

    # Free-flow Markov assignment as initial flow
    freeflow_costs = {lid: links[lid].free_flow_time for lid in links}
    freeflow_flows, _ = assign_all_od_absorbing_markov_by_origin(
        links=links,
        od=od,
        theta=theta,
        generalized_costs=freeflow_costs,
    )

    # Uniform small flow
    uniform_flows = {lid: 100.0 for lid in links}

    # Capacity-proportional flow
    capacity_half_flows = {lid: 0.5 * links[lid].capacity for lid in links}

    # Random initial flow
    np.random.seed(42)
    random_flows = {
        lid: float(np.random.uniform(0.0, links[lid].capacity))
        for lid in links
    }

    return {
        "zero_start": zero_flows,
        "freeflow_assignment_start": freeflow_flows,
        "uniform_100_start": uniform_flows,
        "half_capacity_start": capacity_half_flows,
        "random_start": random_flows,
    }


def compare_bpr_assignment_starting_points():
    links = build_grid_network()
    true_od = build_true_od()
    true_link_flows = build_true_link_flows()[1]
    theta = build_model_parameters()["theta"]

    initial_cases = build_initial_flow_cases(
        links=links,
        od=true_od,
        theta=theta,
    )

    results = {}

    for name, init_flows in initial_cases.items():
        print(f"\nRunning BPR/logit-SUE assignment with starting point: {name}")

        result = run_bpr_logit_equilibrium_assignment(
            links=links,
            od=true_od,
            theta=theta,
            initial_flows=init_flows,
            max_iter=5000,
            tol=1e-8,
            order_name=name,
        )

        results[name] = result

        rmsep = compute_rmsep(result["flows"], true_link_flows)

        print(f"  Iterations: {result['iterations']}")
        print(f"  Last max flow difference: {result['last_diff']:.10f}")
        print(f"  RMSEP vs paper true flows: {rmsep:.4f}")

    # --------------------------------------------------------
    # Build comparison table
    # --------------------------------------------------------

    rows = []

    for lid in sorted(links):
        row = {
            "Link": lid,
            "Paper true flow": true_link_flows[lid],
        }

        for name, result in results.items():
            row[name] = round(result["flows"][lid], 4)

        rows.append(row)

    comparison_table = pd.DataFrame(rows)

    # --------------------------------------------------------
    # Build RMSEP summary table
    # --------------------------------------------------------

    rmsep_rows = []

    for name, result in results.items():
        rmsep_rows.append({
            "Starting point": name,
            "Iterations": result["iterations"],
            "Last max diff": result["last_diff"],
            "RMSEP vs paper true flows": compute_rmsep(result["flows"], true_link_flows),
        })

    rmsep_table = pd.DataFrame(rmsep_rows)

    print("\n===== LINK FLOW COMPARISON ACROSS STARTING POINTS =====")
    display(comparison_table)

    print("\n===== RMSEP SUMMARY =====")
    display(rmsep_table)

    return results, comparison_table, rmsep_table


# ============================================================
# RUN TEST
# ============================================================

bpr_start_results, bpr_start_comparison, bpr_start_rmsep = compare_bpr_assignment_starting_points()


Running BPR/logit-SUE assignment with starting point: zero_start
  Iterations: 5000
  Last max flow difference: 0.0000001685
  RMSEP vs paper true flows: 0.2069

Running BPR/logit-SUE assignment with starting point: freeflow_assignment_start
  Iterations: 5000
  Last max flow difference: 0.0000000167
  RMSEP vs paper true flows: 0.2070

Running BPR/logit-SUE assignment with starting point: uniform_100_start
  Iterations: 5000
  Last max flow difference: 0.0000001743
  RMSEP vs paper true flows: 0.2069

Running BPR/logit-SUE assignment with starting point: half_capacity_start
  Iterations: 5000
  Last max flow difference: 0.0000001787
  RMSEP vs paper true flows: 0.2069

Running BPR/logit-SUE assignment with starting point: random_start
  Iterations: 5000
  Last max flow difference: 0.0000003751
  RMSEP vs paper true flows: 0.2070

===== LINK FLOW COMPARISON ACROSS STARTING POINTS =====


,Link,Paper true flow,zero_start,freeflow_assignment_start,uniform_100_start,half_capacity_start,random_start
0,1,124,123.7284,123.7283,123.7284,123.7284,123.7296
1,2,137,137.2585,137.2585,137.2585,137.2585,137.2576
2,3,109,109.0131,109.0133,109.0131,109.0131,109.0128
3,4,77,77.1642,77.1649,77.1642,77.1642,77.1645
4,5,467,466.5642,466.5634,466.5642,466.5643,466.5651
5,6,77,77.1642,77.1649,77.1642,77.1642,77.1645
6,7,212,211.5714,211.5716,211.5714,211.5714,211.5707
7,8,295,295.6871,295.6869,295.6871,295.6871,295.6869
8,9,303,302.6661,302.6656,302.6662,302.6661,302.6648
9,10,400,399.7152,399.7151,399.7152,399.7152,399.7155



===== RMSEP SUMMARY =====


,Starting point,Iterations,Last max diff,RMSEP vs paper true flows
0,zero_start,5000,1.684579e-07,0.206926
1,freeflow_assignment_start,5000,1.674493e-08,0.206990
2,uniform_100_start,5000,1.743189e-07,0.206916
3,half_capacity_start,5000,1.786775e-07,0.206925
4,random_start,5000,3.751325e-07,0.207040


In [16]:
# ============================================================
# STEP 1 — TRUNCATED / FINITE-HORIZON MARKOV ASSIGNMENT
# ============================================================
#
# IDEA:
# Instead of allowing infinite propagation:
#
#   V = I + W + W² + W³ + ...
#
# we truncate after K transitions:
#
#   V_K = I + W + W² + ... + W^K
#
# This limits unrealistic very long paths.
#
# ============================================================

from collections import deque
import math


# ============================================================
# TRUNCATED VALUE FUNCTION
# ============================================================

def compute_destination_value_functions_truncated(
    links,
    theta,
    destination,
    max_depth=5,
    generalized_costs=None,
):
    """
    Finite-horizon destination attractiveness/value function.

    max_depth:
        maximum number of link transitions allowed.
    """

    if generalized_costs is None:
        generalized_costs = {
            lid: links[lid].free_flow_time
            for lid in links
        }

    outgoing, _ = build_adjacency(links)

    # --------------------------------------------------------
    # A[(node, depth)] = attractiveness
    # --------------------------------------------------------

    A = {}

    # Base condition:
    # destination has attractiveness 1 at all depths
    nodes = get_nodes_from_links(links)

    for d in range(max_depth + 1):
        A[(destination, d)] = 1.0

    # Other nodes initialize to zero
    for node in nodes:
        for d in range(max_depth + 1):
            if (node, d) not in A:
                A[(node, d)] = 0.0

    # --------------------------------------------------------
    # Backward recursion
    # --------------------------------------------------------

    for depth in reversed(range(max_depth)):

        for node in nodes:

            if node == destination:
                continue

            total = 0.0

            for lid in outgoing.get(node, []):

                next_node = links[lid].to_node

                total += (
                    math.exp(-theta * generalized_costs[lid])
                    * A[(next_node, depth + 1)]
                )

            A[(node, depth)] = total

    # Final node attractiveness uses depth=0
    final_A = {
        node: A[(node, 0)]
        for node in nodes
    }

    return final_A


# ============================================================
# TRUNCATED TRANSITION PROBABILITIES
# ============================================================

def compute_transition_probabilities_truncated(
    links,
    theta,
    destination,
    max_depth=5,
    generalized_costs=None,
):

    if generalized_costs is None:
        generalized_costs = {
            lid: links[lid].free_flow_time
            for lid in links
        }

    outgoing, _ = build_adjacency(links)

    A = compute_destination_value_functions_truncated(
        links=links,
        theta=theta,
        destination=destination,
        max_depth=max_depth,
        generalized_costs=generalized_costs,
    )

    probs = {}

    for node in get_nodes_from_links(links):

        denom = A[node]

        if denom <= 0:
            continue

        for lid in outgoing.get(node, []):

            next_node = links[lid].to_node

            downstream = (
                1.0
                if next_node == destination
                else A[next_node]
            )

            if downstream <= 0:
                continue

            numer = (
                math.exp(-theta * generalized_costs[lid])
                * downstream
            )

            probs[lid] = numer / denom

    return probs, A


# ============================================================
# TRUNCATED SINGLE OD ASSIGNMENT
# ============================================================

def assign_single_od_markov_truncated(
    links,
    origin,
    destination,
    demand,
    theta,
    max_depth=5,
    generalized_costs=None,
):

    if generalized_costs is None:
        generalized_costs = {
            lid: links[lid].free_flow_time
            for lid in links
        }

    probs, A = compute_transition_probabilities_truncated(
        links=links,
        theta=theta,
        destination=destination,
        max_depth=max_depth,
        generalized_costs=generalized_costs,
    )

    outgoing, _ = build_adjacency(links)
    order = topological_order(links)

    node_flow = {
        node: 0.0
        for node in get_nodes_from_links(links)
    }

    link_flow = {
        lid: 0.0
        for lid in links
    }

    node_flow[origin] = demand

    for node in order:

        if node == destination:
            continue

        inflow = node_flow[node]

        if inflow <= 0:
            continue

        for lid in outgoing.get(node, []):

            if lid not in probs:
                continue

            f = inflow * probs[lid]

            next_node = links[lid].to_node

            link_flow[lid] += f
            node_flow[next_node] += f

    return link_flow, node_flow, probs, A


# ============================================================
# ASSIGN ALL OD PAIRS
# ============================================================

def assign_all_od_markov_truncated(
    links,
    od,
    theta,
    max_depth=5,
    generalized_costs=None,
):

    if generalized_costs is None:
        generalized_costs = {
            lid: links[lid].free_flow_time
            for lid in links
        }

    total_link_flows = {
        lid: 0.0
        for lid in links
    }

    for (r, s), q_rs in od.items():

        link_flow, _, _, _ = assign_single_od_markov_truncated(
            links=links,
            origin=r,
            destination=s,
            demand=q_rs,
            theta=theta,
            max_depth=max_depth,
            generalized_costs=generalized_costs,
        )

        for lid, flow in link_flow.items():
            total_link_flows[lid] += flow

    return total_link_flows


# ============================================================
# TEST DIFFERENT K VALUES
# ============================================================

def test_truncated_assignment():

    links = build_grid_network()
    true_od = build_true_od()
    true_flows = build_true_link_flows()[1]

    theta = build_model_parameters()["theta"]

    K_values = [1, 2, 3, 4, 5, 6, 10]

    results = []

    print("\n===== TRUNCATED MARKOV ASSIGNMENT =====")

    for K in K_values:

        est_flows = assign_all_od_markov_truncated(
            links=links,
            od=true_od,
            theta=theta,
            max_depth=K,
        )

        rmsep = compute_rmsep(est_flows, true_flows)

        print(f"\nK = {K}")
        print(f"RMSEP = {rmsep:.4f}")

        results.append({
            "K": K,
            "RMSEP": rmsep,
        })

    return results


# ============================================================
# RUN
# ============================================================

truncated_results = test_truncated_assignment()


===== TRUNCATED MARKOV ASSIGNMENT =====

K = 1
RMSEP = 100.0000

K = 2
RMSEP = 299.6001

K = 3
RMSEP = 23.0116

K = 4
RMSEP = 5.3480

K = 5
RMSEP = 5.3480

K = 6
RMSEP = 5.3480

K = 10
RMSEP = 5.3480


In [21]:
# ============================================================
# FLOW TABLE FOR EACH LINK UNDER EACH K
# ============================================================

import pandas as pd


def build_truncated_flow_table():
    links = build_grid_network()
    true_od = build_true_od()
    true_flows = build_true_link_flows()[1]
    theta = build_model_parameters()["theta"]

    K_values = [1, 2, 3, 4, 5, 6, 1000]

    rows = []

    # --------------------------------------------------------
    # Compute estimated link flows for each K
    # --------------------------------------------------------

    flow_results = {}

    for K in K_values:
        est_flows = assign_all_od_markov_truncated(
            links=links,
            od=true_od,
            theta=theta,
            max_depth=K,
        )

        flow_results[K] = est_flows

    # --------------------------------------------------------
    # Build link-by-link table
    # --------------------------------------------------------

    for lid in sorted(links):

        row = {
            "Link": lid,
            "Paper true flow": true_flows[lid],
        }

        for K in K_values:
            row[f"K={K}"] = round(flow_results[K][lid], 2)

        rows.append(row)

    flow_table = pd.DataFrame(rows)

    # --------------------------------------------------------
    # Build RMSEP table
    # --------------------------------------------------------

    rmsep_rows = []

    for K in K_values:
        rmsep_rows.append({
            "K": K,
            "RMSEP": round(compute_rmsep(flow_results[K], true_flows), 4),
        })

    rmsep_table = pd.DataFrame(rmsep_rows)

    print("\n===== LINK FLOW TABLE BY K =====")
    display(flow_table)

    print("\n===== RMSEP BY K =====")
    display(rmsep_table)

    return flow_table, rmsep_table, flow_results


# ============================================================
# RUN
# ============================================================

truncated_flow_table, truncated_rmsep_table, truncated_flow_results = build_truncated_flow_table()


===== LINK FLOW TABLE BY K =====


,Link,Paper true flow,K=1,K=2,K=3,K=4,K=5,K=6,K=1000
0,1,124,0.0,426.68,161.02,124.57,124.57,124.57,124.57
1,2,137,0.0,492.33,187.15,142.74,142.74,142.74,142.74
2,3,109,0.0,517.24,136.11,102.68,102.68,102.68,102.68
3,4,77,0.0,140.79,74.06,71.05,71.05,71.05,71.05
4,5,467,0.0,1028.04,506.95,473.53,473.53,473.53,473.53
5,6,77,0.0,140.79,74.06,71.05,71.05,71.05,71.05
6,7,212,0.0,629.12,216.24,200.45,200.45,200.45,200.45
7,8,295,0.0,1169.40,340.91,312.29,312.29,312.29,312.29
8,9,303,0.0,676.24,321.89,306.10,306.10,306.10,306.10
9,10,400,0.0,1050.93,425.53,392.10,392.10,392.10,392.10



===== RMSEP BY K =====


,K,RMSEP
0,1,100.0000
1,2,299.6001
2,3,23.0116
3,4,5.3480
4,5,5.3480
5,6,5.3480
6,1000,5.3480


In [24]:
# ============================================================
# SAFE BPR / LOGIT-SUE WITH TRUNCATED MARKOV K
# SKIP UNSTABLE VERY SMALL K VALUES
# ============================================================

import pandas as pd
import numpy as np
import math


def safe_compute_rmsep(estimated, true_values):
    squared_pct_errors = []

    for lid, x_true in true_values.items():
        if x_true == 0:
            continue

        x_hat = estimated[lid]

        # Guard against numerical explosion
        if not np.isfinite(x_hat) or abs(x_hat) > 1e12:
            return np.nan

        err = (x_hat - x_true) / x_true
        squared_pct_errors.append(err ** 2)

    return 100.0 * math.sqrt(sum(squared_pct_errors) / len(squared_pct_errors))


def test_bpr_truncated_K_values_safe_v2():
    links = build_grid_network()
    true_od = build_true_od()
    true_flows = build_true_link_flows()[1]
    theta = build_model_parameters()["theta"]

    # K=1 and K=2 are structurally too short for this network
    K_values = [3, 4, 5, 6, 10, 1000]

    results = {}

    print("\n===== SAFE BPR / LOGIT-SUE WITH TRUNCATED MARKOV K =====")

    for K in K_values:
        print(f"\nRunning K = {K}")

        try:
            result = run_bpr_logit_equilibrium_assignment_truncated_K_safe(
                links=links,
                od=true_od,
                theta=theta,
                K=K,
                initial_flows=None,
                max_iter=5000,
                tol=1e-8,
            )

            results[K] = result
            rmsep = safe_compute_rmsep(result["flows"], true_flows)

            print(f"  Iterations: {result['iterations']}")
            print(f"  Last max flow difference: {result['last_diff']:.10f}")
            print(f"  RMSEP: {rmsep:.4f}")

        except Exception as e:
            print(f"  Failed for K={K}: {e}")
            results[K] = None

    # --------------------------------------------------------
    # Link-flow table
    # --------------------------------------------------------

    rows = []

    for lid in sorted(links):
        row = {
            "Link": lid,
            "Paper true flow": true_flows[lid],
        }

        for K in K_values:
            if results[K] is None:
                row[f"K={K}"] = np.nan
            else:
                flow_val = results[K]["flows"][lid]
                row[f"K={K}"] = round(flow_val, 2) if np.isfinite(flow_val) else np.nan

        rows.append(row)

    flow_table = pd.DataFrame(rows)

    # --------------------------------------------------------
    # RMSEP table
    # --------------------------------------------------------

    rmsep_rows = []

    for K in K_values:
        if results[K] is None:
            rmsep_rows.append({
                "K": K,
                "Iterations": None,
                "Last max diff": None,
                "RMSEP": np.nan,
            })
        else:
            rmsep_rows.append({
                "K": K,
                "Iterations": results[K]["iterations"],
                "Last max diff": results[K]["last_diff"],
                "RMSEP": round(safe_compute_rmsep(results[K]["flows"], true_flows), 4),
            })

    rmsep_table = pd.DataFrame(rmsep_rows)

    print("\n===== LINK FLOW TABLE: SAFE BPR + TRUNCATED K =====")
    display(flow_table)

    print("\n===== RMSEP TABLE: SAFE BPR + TRUNCATED K =====")
    display(rmsep_table)

    return results, flow_table, rmsep_table


bpr_truncated_results, bpr_truncated_flow_table, bpr_truncated_rmsep_table = test_bpr_truncated_K_values_safe_v2()


===== SAFE BPR / LOGIT-SUE WITH TRUNCATED MARKOV K =====

Running K = 3
  Iterations: 5000
  Last max flow difference: 0.0000001539
  RMSEP: 21.0602

Running K = 4
  Iterations: 5000
  Last max flow difference: 0.0000001685
  RMSEP: 0.2069

Running K = 5
  Iterations: 5000
  Last max flow difference: 0.0000001685
  RMSEP: 0.2069

Running K = 6
  Iterations: 5000
  Last max flow difference: 0.0000001685
  RMSEP: 0.2069

Running K = 10
  Iterations: 5000
  Last max flow difference: 0.0000001685
  RMSEP: 0.2069

Running K = 1000
  Iterations: 5000
  Last max flow difference: 0.0000001685
  RMSEP: 0.2069

===== LINK FLOW TABLE: SAFE BPR + TRUNCATED K =====


,Link,Paper true flow,K=3,K=4,K=5,K=6,K=10,K=1000
0,1,124,154.44,123.73,123.73,123.73,123.73,123.73
1,2,137,169.57,137.26,137.26,137.26,137.26,137.26
2,3,109,140.85,109.01,109.01,109.01,109.01,109.01
3,4,77,82.10,77.16,77.16,77.16,77.16,77.16
4,5,467,492.35,466.56,466.56,466.56,466.56,466.56
5,6,77,82.10,77.16,77.16,77.16,77.16,77.16
6,7,212,229.31,211.57,211.57,211.57,211.57,211.57
7,8,295,310.26,295.69,295.69,295.69,295.69,295.69
8,9,303,317.94,302.67,302.67,302.67,302.67,302.67
9,10,400,424.51,399.72,399.72,399.72,399.72,399.72



===== RMSEP TABLE: SAFE BPR + TRUNCATED K =====


,K,Iterations,Last max diff,RMSEP
0,3,5000,1.538842e-07,21.0602
1,4,5000,1.684579e-07,0.2069
2,5,5000,1.684579e-07,0.2069
3,6,5000,1.684579e-07,0.2069
4,10,5000,1.684579e-07,0.2069
5,1000,5000,1.684579e-07,0.2069


In [18]:
# ============================================================
# OD ORDER SENSITIVITY TEST
# FIXED-COST ABSORBING MARKOV ASSIGNMENT
# ============================================================

import pandas as pd
import random


def assign_all_od_with_custom_order(
    links,
    od,
    theta,
    od_order,
    generalized_costs=None,
):
    """
    Assign OD pairs using a user-defined order.
    """

    if generalized_costs is None:
        generalized_costs = {
            lid: links[lid].free_flow_time
            for lid in links
        }

    total_link_flows = {
        lid: 0.0
        for lid in links
    }

    origin_link_flows = {}

    for od_pair in od_order:

        r, s = od_pair
        q_rs = od[od_pair]

        # ----------------------------------------------------
        # YOUR NOTEBOOK FUNCTION
        # ----------------------------------------------------
        link_flow, _, _, _ = assign_single_od_absorbing_markov(
            links=links,
            origin=r,
            destination=s,
            demand=q_rs,
            theta=theta,
            generalized_costs=generalized_costs,
        )

        # ----------------------------------------------------
        # ADD FLOWS TO TOTAL
        # ----------------------------------------------------
        for lid, flow in link_flow.items():
            total_link_flows[lid] += flow

    return total_link_flows


def test_od_order_sensitivity():

    links = build_grid_network()

    true_od = build_true_od()

    true_flows = build_true_link_flows()[1]

    params = build_model_parameters()

    theta = params["theta"]

    # ========================================================
    # DIFFERENT OD ORDERS
    # ========================================================

    original_order = list(true_od.keys())

    reverse_order = list(reversed(original_order))

    by_destination_order = sorted(
        original_order,
        key=lambda x: (x[1], x[0])
    )

    by_origin_order = sorted(
        original_order,
        key=lambda x: (x[0], x[1])
    )

    random.seed(42)

    random_order = original_order.copy()

    random.shuffle(random_order)

    orders = {
        "original_order": original_order,
        "reverse_order": reverse_order,
        "by_destination_order": by_destination_order,
        "by_origin_order": by_origin_order,
        "random_order": random_order,
    }

    # ========================================================
    # RUN ASSIGNMENT FOR EACH ORDER
    # ========================================================

    results = {}

    print("\n===== OD ORDER SENSITIVITY TEST =====")

    for name, od_order in orders.items():

        flows = assign_all_od_with_custom_order(
            links=links,
            od=true_od,
            theta=theta,
            od_order=od_order,
        )

        rmsep = compute_rmsep(flows, true_flows)

        results[name] = {
            "flows": flows,
            "rmsep": rmsep,
        }

        print(f"\n{name}")
        print("OD sequence:")

        for pair in od_order:
            print(f"  {pair}: {true_od[pair]}")

        print(f"RMSEP = {rmsep:.6f}")

    # ========================================================
    # LINK FLOW COMPARISON TABLE
    # ========================================================

    rows = []

    for lid in sorted(links):

        row = {
            "Link": lid,
            "Paper true flow": true_flows[lid],
        }

        for name in orders:
            row[name] = round(results[name]["flows"][lid], 6)

        rows.append(row)

    comparison_table = pd.DataFrame(rows)

    # ========================================================
    # DIFFERENCE SUMMARY
    # ========================================================

    original_flows = results["original_order"]["flows"]

    diff_rows = []

    for name in orders:

        if name == "original_order":
            continue

        max_abs_diff = max(
            abs(results[name]["flows"][lid] - original_flows[lid])
            for lid in links
        )

        diff_rows.append({
            "Order": name,
            "RMSEP": round(results[name]["rmsep"], 6),
            "Max abs diff from original": max_abs_diff,
        })

    diff_table = pd.DataFrame(diff_rows)

    # ========================================================
    # DISPLAY RESULTS
    # ========================================================

    print("\n===== LINK FLOW COMPARISON =====")

    display(comparison_table)

    print("\n===== DIFFERENCE SUMMARY =====")

    display(diff_table)

    return results, comparison_table, diff_table


# ============================================================
# RUN TEST
# ============================================================

order_results, comparison_table, diff_table = test_od_order_sensitivity()


===== OD ORDER SENSITIVITY TEST =====

original_order
OD sequence:
  (1, 6): 120.0
  (1, 8): 150.0
  (1, 9): 100.0
  (2, 6): 130.0
  (2, 8): 200.0
  (2, 9): 90.0
  (4, 6): 80.0
  (4, 8): 180.0
  (4, 9): 110.0
RMSEP = 5.348029

reverse_order
OD sequence:
  (4, 9): 110.0
  (4, 8): 180.0
  (4, 6): 80.0
  (2, 9): 90.0
  (2, 8): 200.0
  (2, 6): 130.0
  (1, 9): 100.0
  (1, 8): 150.0
  (1, 6): 120.0
RMSEP = 5.348029

by_destination_order
OD sequence:
  (1, 6): 120.0
  (2, 6): 130.0
  (4, 6): 80.0
  (1, 8): 150.0
  (2, 8): 200.0
  (4, 8): 180.0
  (1, 9): 100.0
  (2, 9): 90.0
  (4, 9): 110.0
RMSEP = 5.348029

by_origin_order
OD sequence:
  (1, 6): 120.0
  (1, 8): 150.0
  (1, 9): 100.0
  (2, 6): 130.0
  (2, 8): 200.0
  (2, 9): 90.0
  (4, 6): 80.0
  (4, 8): 180.0
  (4, 9): 110.0
RMSEP = 5.348029

random_order
OD sequence:
  (2, 6): 130.0
  (4, 6): 80.0
  (4, 8): 180.0
  (2, 8): 200.0
  (4, 9): 110.0
  (1, 9): 100.0
  (2, 9): 90.0
  (1, 6): 120.0
  (1, 8): 150.0
RMSEP = 5.348029

===== LINK FLOW 

,Link,Paper true flow,original_order,reverse_order,by_destination_order,by_origin_order,random_order
0,1,124,124.573329,124.573329,124.573329,124.573329,124.573329
1,2,137,142.742362,142.742362,142.742362,142.742362,142.742362
2,3,109,102.684309,102.684309,102.684309,102.684309,102.684309
3,4,77,71.045887,71.045887,71.045887,71.045887,71.045887
4,5,467,473.527442,473.527442,473.527442,473.527442,473.527442
5,6,77,71.045887,71.045887,71.045887,71.045887,71.045887
6,7,212,200.449387,200.449387,200.449387,200.449387,200.449387
7,8,295,312.292975,312.292975,312.292975,312.292975,312.292975
8,9,303,306.104164,306.104164,306.104164,306.104164,306.104164
9,10,400,392.100506,392.100506,392.100506,392.100506,392.100506



===== DIFFERENCE SUMMARY =====


,Order,RMSEP,Max abs diff from original
0,reverse_order,5.348029,2.842171e-14
1,by_destination_order,5.348029,5.684342e-14
2,by_origin_order,5.348029,0.000000e+00
3,random_order,5.348029,1.136868e-13
